## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [21]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import (
    Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel,
    input_guardrail, output_guardrail, GuardrailFunctionOutput, RunContextWrapper,
    OutputGuardrailTripwireTriggered,
)
from typing import Dict, List, Optional
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel, Field

In [22]:
load_dotenv(override=True)

True

In [23]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
DeepSeek API Key exists and begins sk-
Groq API Key exists and begins gsk_


In [24]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

### It's easy to use any models with OpenAI compatible endpoints

In [25]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [26]:

deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

In [27]:
sales_agent1 = Agent(name="DeepSeek Sales Agent", instructions=instructions1, model=deepseek_model)
sales_agent2 =  Agent(name="Gemini Sales Agent", instructions=instructions2, model=gemini_model)
sales_agent3  = Agent(name="Llama3.3 Sales Agent",instructions=instructions3,model=llama3_3_model)

In [28]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [29]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("sankari.s2009@gmail.com")  # Change to your verified sender
    to_email = To("sankari.s2009@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [30]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

#Exercise

class SubjectSuggestion(BaseModel):
    
    subject: str = Field(description="A compelling subject line for the email")

subject_writer = Agent(
    name="Email subject writer",
    instructions=subject_instructions,
    model="gpt-4o-mini",
    output_type=SubjectSuggestion,
)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [31]:
email_tools = [subject_tool, html_tool, send_html_email]

In [32]:
email_tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001AB25315EE0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001AB25316980>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=No

In [33]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")

In [34]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [35]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

## Check out the trace:

https://platform.openai.com/traces

In [36]:
# Exercise - more structured output
class NameCheckOutput(BaseModel):
   
    is_name_in_message: bool
    name: str


class IntentCheckOutput(BaseModel):
  
    is_sales_email_request: bool
    intent_summary: str = Field(description="Short label for the detected intent")
    rejection_reason: Optional[str] = Field(default=None, description="If not a sales email request, why not")


class EmailContentCheckOutput(BaseModel):
   
    is_appropriate: bool
    issues: List[str] = Field(default_factory=list, description="List of issues found, if any")
    suggested_fix: Optional[str] = None



class ColdEmailDraft(BaseModel):
   
    subject: str = Field(description="Email subject line")
    body: str = Field(description="Plain text email body")



guardrail_agent = Agent(
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini",
)

intent_check_agent = Agent(
    name="Intent check",
    instructions="Determine if the user is asking to send a cold sales email for ComplAI (SOC2 compliance SaaS). "
    "Return is_sales_email_request=true only when the request is clearly about composing/sending a cold sales or outreach email. "
    "Return false for unrelated requests (e.g. general chat, other tasks).",
    output_type=IntentCheckOutput,
    model="gpt-4o-mini",
)

email_content_check_agent = Agent(
    name="Email content check",
    instructions="Review an email draft before it is sent. Check that it is professional, on-topic (ComplAI/SOC2), "
    "and does not contain inappropriate language, aggressive tone, or obvious PII that should not be in the body. "
    "List any issues found; if none, set is_appropriate=true and leave issues empty.",
    output_type=EmailContentCheckOutput,
    model="gpt-4o-mini",
)

In [37]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output}, tripwire_triggered=is_name_in_message)


@input_guardrail
async def guardrail_against_bad_intent(ctx, agent, message):
   
    result = await Runner.run(intent_check_agent, message, context=ctx.context)
    out = result.final_output
    return GuardrailFunctionOutput(
        output_info={"intent_check": out},
        tripwire_triggered=not out.is_sales_email_request,
    )

In [38]:
@output_guardrail
async def guardrail_email_content(ctx: RunContextWrapper, agent: Agent, output) -> GuardrailFunctionOutput:
    
    draft_text = output if isinstance(output, str) else str(output)
    result = await Runner.run(email_content_check_agent, draft_text, context=ctx.context)
    out = result.final_output
    return GuardrailFunctionOutput(
        output_info={"email_content_check": out},
        tripwire_triggered=not out.is_appropriate,
    )

### Testing the new guardrails



In [41]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name, guardrail_against_bad_intent],
    output_guardrails=[guardrail_email_content],
)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

## Check out the trace:

https://platform.openai.com/traces

In [42]:

message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

### Trigger output guardrail



In [44]:
# Draft that should fail the content check 
BAD_EMAIL_DRAFT = (
    "You need to buy our SOC2 tool RIGHT NOW or your company will fail the audit. "
    "We are the best. Reply immediately or we will assume you are not serious. "
    "Contact us today or regret it."
)


check_result = await Runner.run(email_content_check_agent, BAD_EMAIL_DRAFT)
print("Content check on bad draft:", check_result.final_output)
assert not check_result.final_output.is_appropriate, "Expected is_appropriate=False"


output_guardrail_demo_agent = Agent(
    name="Bad draft demo",
    instructions="When the user says 'send bad draft', write a short cold email that is deliberately "
    "aggressive and unprofessional: demand the reader act immediately, use threatening or pushy language, "
    "and sound like you are pressuring them. Output ONLY the email body, nothing else.",
    model="gpt-4o-mini",
    output_guardrails=[guardrail_email_content],
)
try:
    with trace("Trigger output guardrail"):
        await Runner.run(output_guardrail_demo_agent, "send bad draft")
    print("Unexpected: output guardrail did not trip")
except OutputGuardrailTripwireTriggered as e:
    print("Output guardrail triggered (expected):", type(e).__name__)

Content check on bad draft: is_appropriate=False issues=['Use of aggressive and urgent language', 'Inappropriate tone and pressure tactics', 'Lack of professionalism'] suggested_fix='Revise the email to emphasize the benefits of the tool and invite questions or discussions in a more professional manner.'


NameError: name 'OutputGuardrailTripwireTriggered' is not defined

In [45]:
# Test intent guardrail: non–sales-email request should be blocked 
try:
    with trace("Test intent guardrail"):
        await Runner.run(careful_sales_manager, "What's the weather in London today?")
except Exception as e:
    print(f"Intent guardrail triggered (expected): {type(e).__name__}")

Intent guardrail triggered (expected): InputGuardrailTripwireTriggered
